<a href="https://colab.research.google.com/github/antipetajyothsna-dev/Sales-Trends-Visualization-Project-/blob/main/Sales_Trend_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
"""
Sales Trend Visualization Script
Author: Gemini Data Analysis Assistant
Description: Cleans a standard sales dataset, aggregates data across time,
             and exports professional data visualizations for presentation.
"""

import os
import pandas as pd
import matplotlib.pyplot as plt

# Set style globally for clean, modern charts
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

def load_and_preprocess_data(file_path: str) -> pd.DataFrame:
    """
    Loads the CSV file and ensures correct data types and time features.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file '{file_path}' was not found. Please verify the path.")

    print("Loading and preparing dataset...")
    df = pd.read_csv(file_path)

    # Standardize 'Order Date' to datetime objects
    df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')

    # Extract structural time components
    df['Year'] = df['Order Date'].dt.year
    df['Month'] = df['Order Date'].dt.month
    df['YearMonth'] = df['Order Date'].dt.to_period('M')

    return df


def generate_overall_trend(df: pd.DataFrame, output_name: str = 'overall_sales_trend.png'):
    """
    Aggregates overall monthly revenue and saves a sequential line chart.
    """
    print(f"Generating overall monthly sales trend -> saving to {output_name}")

    # Aggregate data
    monthly_sales = df.groupby('YearMonth')['Sales'].sum().reset_index()
    monthly_sales['YearMonth'] = monthly_sales['YearMonth'].dt.to_timestamp()

    # Plot using subplots to avoid using .figure()
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(
        monthly_sales['YearMonth'],
        monthly_sales['Sales'],
        marker='o',
        color='#1f77b4',
        linewidth=2,
        markersize=5
    )

    # Styling and formatting
    ax.set_title('Overall Monthly Sales Trend (2015-2018)', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Order Date', fontsize=11, labelpad=10)
    ax.set_ylabel('Total Sales ($)', fontsize=11, labelpad=10)
    ax.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(output_name, dpi=300)
    plt.close(fig)


def generate_category_trend(df: pd.DataFrame, output_name: str = 'sales_trend_by_category.png'):
    """
    Aggregates monthly revenue grouped by Product Category and saves the chart.
    """
    print(f"Generating categorical sales trends -> saving to {output_name}")

    # Aggregate data
    category_sales = df.groupby(['YearMonth', 'Category'])['Sales'].sum().reset_index()
    category_sales['YearMonth'] = category_sales['YearMonth'].dt.to_timestamp()

    fig, ax = plt.subplots(figsize=(12, 6))

    # Plot a distinct line for each product category
    colors = ['#ff7f0e', '#2ca02c', '#d62728']
    for idx, cat in enumerate(category_sales['Category'].unique()):
        cat_data = category_sales[category_sales['Category'] == cat]
        ax.plot(
            cat_data['YearMonth'],
            cat_data['Sales'],
            marker='s',
            markersize=4,
            linewidth=2,
            label=cat,
            color=colors[idx % len(colors)]
        )

    ax.set_title('Monthly Sales Trend by Product Category', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Order Date', fontsize=11, labelpad=10)
    ax.set_ylabel('Total Sales ($)', fontsize=11, labelpad=10)
    ax.legend(title='Product Category', frameon=True, loc='upper left')
    ax.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(output_name, dpi=300)
    plt.close(fig)


def generate_seasonality_chart(df: pd.DataFrame, output_name: str = 'sales_seasonality.png'):
    """
    Creates an overlay chart comparing monthly cyclical sales patterns across separate years.
    """
    print(f"Generating annual seasonality overlay -> saving to {output_name}")

    # Aggregate data by year and calendar month number
    seasonal_sales = df.groupby(['Year', 'Month'])['Sales'].sum().reset_index()
    pivot_seasonal = seasonal_sales.pivot(index='Month', columns='Year', values='Sales')

    fig, ax = plt.subplots(figsize=(11, 6))

    # Plot using built-in pandas line mapping on the explicit axis object
    pivot_seasonal.plot(kind='line', marker='o', ax=ax, linewidth=2, colormap='viridis')

    ax.set_title('Sales Seasonality: Comparison Across Calendar Years', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Month of the Year', fontsize=11, labelpad=10)
    ax.set_ylabel('Total Sales ($)', fontsize=11, labelpad=10)

    # Explicitly configure calendar x-ticks from 1 to 12
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(title='Year', loc='upper left')

    plt.tight_layout()
    plt.savefig(output_name, dpi=300)
    plt.close(fig)


def display_growth_metrics(df: pd.DataFrame):
    """
    Computes and logs simple summary growth data to the console terminal.
    """
    print("\n" + "="*40)
    print("        ANNUAL PERFORMANCE SUMMARY")
    print("="*40)

    yearly_sales = df.groupby('Year')['Sales'].sum().reset_index()
    yearly_sales['YoY Growth (%)'] = yearly_sales['Sales'].pct_change() * 100

    for _, row in yearly_sales.iterrows():
        growth_str = f"{row['YoY Growth (%)']:.2f}%" if not pd.isna(row['YoY Growth (%)']) else "Baseline Year"
        print(f"Year {int(row['Year'])}: Total Sales = ${row['Sales']:,.2f} | YoY Growth = {growth_str}")
    print("="*40 + "\n")


def main():
    # Configure path pointing directly to your local file
    data_file = 'train.csv'

    try:
        # Execution workflow pipeline
        sales_data = load_and_preprocess_data(data_file)

        # Display operational metrics
        display_growth_metrics(sales_data)

        # Build out visualization asset generation
        generate_overall_trend(sales_data)
        generate_category_trend(sales_data)
        generate_seasonality_chart(sales_data)

        print("Success! All visualization plots have been cleanly generated in your folder.")

    except Exception as e:
        print(f"An error occurred during pipeline execution: {e}")


if __name__ == "__main__":
    main()

Loading and preparing dataset...

        ANNUAL PERFORMANCE SUMMARY
Year 2015: Total Sales = $479,856.21 | YoY Growth = Baseline Year
Year 2016: Total Sales = $459,436.01 | YoY Growth = -4.26%
Year 2017: Total Sales = $600,192.55 | YoY Growth = 30.64%
Year 2018: Total Sales = $722,052.02 | YoY Growth = 20.30%

Generating overall monthly sales trend -> saving to overall_sales_trend.png
Generating categorical sales trends -> saving to sales_trend_by_category.png
Generating annual seasonality overlay -> saving to sales_seasonality.png
Success! All visualization plots have been cleanly generated in your folder.
